<a href="https://colab.research.google.com/github/DanielHevdeli/hafifot-tiug/blob/main/LLM_as_annotator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/DanielHevdeli/hafifot-tiug.git

In [ ]:
pip install dspy

In [3]:
import pandas as pd
import os
from typing import Literal, List
import dspy
import requests
import json

In [4]:
posts_df = pd.read_csv('./hafifot-tiug/data/split_data/present.csv')

In [5]:
posts_df.head(2)

,question_id,length,date,text
0,114678,903,2016-04-10 20:05:00,"שלום , אני מאוד מקווה שתוכלו לעזור לי אני לא י..."
1,116349,888,2016-04-24 19:56:00,היי כולם \nיש לי בעיה הקשורה לתספורת שלי. \nאמ...


In [6]:
posts_df.iloc[3]['text']

'בערך מגיל 17 אני סובלת מחרדה, כאשר חלה החרפה קיצונית בגיל 18 עקב לימודים ומצב חברתי גרוע והשפלות יומיומיות  . בהתחלה לא הבנתי בכלל שמדובר בחרדה כי הסימנים היו פיזיולוגים והחששות תמיד היו. התחלתי "להתעלף"  , הרגשתי שאני לא מצליחה לנשום וגם כשאני כן מצליחה אז האוויר לא נכנס לי לריאות , היו לי סחרחורות שהייתי בטוחה שאני עומדת למות ולהתמוטט. נשלחתי למיון וכאשר לא מצאו לי כלום הניחו שזה חרדה.ככה העברתי את השנה האחרונה שלי בבית ספר, כיתה יב. כלומר, במיטה ובדיכאון ללא חברים ולומדת לבגרויות. \nכמה חודשים אחר כך עליתי על מדים והאמת שמצבי השתפר פלאים. אומנם החרדה לא נעלמה , ותמיד היו ניצוצות של חרדה אבל הצלחתי לעבור אותן. \nכהשתחררתי היה לי מין תהום ענקית , הרגשתי בחופש ורציתי כל היום לישון,מה שבצבא לא התאפשר לי וגם כן לא לפני . יצאתי מדי פעם עם חברות אבל גם זה אחכ הפסיק. אחרי כמה חודשים כאלה התחלתי את הפסיכומטרי ולצערי בגלל תחושות של חרדה והתמוטטות בשיעור דחיתי את המועד. אחרי הדחייה נכנסתי לדכאון וחרדה קשים , לא רציתי לצאת מהמיטה.אם פעם התקפים באו והלכו אז המצב שלי בחודשיים האחרונים היו 24/7. 

In [7]:
posts_df.iloc[2]['text']

'הגעתי לאתר הזה במקרה אחרי שקראתי טיפה מפוסטים של אנונימים פה—אני מקווה שגם אוכל למצוא פה מענה או כיוון לפתרון לבעיה שלי.  \nאני לא יודעת כל כך מאיפה להתחיל או מאיפה התחיל הסיפור אבל אני סובלת בערך כל חיי מחרדה ודיכאון. \nהייתי מודעת לעניין אבל מעולם לא חוויתי מצב קיצוני של חרדה משתקת לחלוטין. היו לי התקפים מזעזעים שבגללם הלכתי למיון.  \nאבל בקיצור, בתקופה האחרונה של כמעט שנה אני חוויתי אגרופוביה רצינית עם התקפי דכאון ומחשבות מזעזעות. ברמה של סיעוד. לא התקלחתי, כל היום הייתי ישנה ומסוגרת מתחת לשמיכה. \nהפסיכאטר נתן לי ציפרלקס .  \nאני מרגישה שיפור עם הדיכאון יחסית, ועם ההתקפי חרדה אבל עד היום האגרובופיה לא נעלמה, קשה לי לתפקד. אני רוצה לצאת החוצה,להכיר אנשים, לעבוד, לקבל רישיון, ללמוד. לא להיות למה שהפכתי. \nמיותר לציין,שניתקתי קשר עם כל סבוביי כך שאין לי חיי חברה.  \nאת כל הכסף שאין לי אני מוציאה על מוניות בדרך לפסיכאטר ובחזרה מימנו כי קשה לי לעלות על אוטובוסים. \nהפסקתי עם השיעורי נהיגה בגלל ההתקפים בנהיגה .  \nבקיצור,אם המצב ימשיך אני לא יודעת איך אוכל לשרוד ככה,גם מבחינה כספית וגם 

Let's try to classify each post to either **suicidal-risk** or **non-suicidal-risk**. It may help the publishers of the website to offer first-help to writers of post categorized as suicidal even before other people answer them.

# Annotate Posts

# wait

In [8]:
class SRClassification(dspy.Signature):
    text: str = dspy.InputField(desc="Hebrew post to classify.")
    label: Literal["suicidal-risk", "non-suicidal-risk"] = dspy.OutputField(
        desc="Classification result."
    )

In [9]:
class SRClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.Predict(SRClassification)

    def forward(self, text: str) -> str:
        result = self.predict(text=text)
        return result.label

# Local LM

In [10]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct" # "ai-forever/mGPT"

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  device_map="auto",
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [30]:
class TinyHebrewLM(dspy.LM):
    def __init__(self, model, tokenizer):
        super().__init__("TinyHebrewLM")
        self._hf_model = model
        self.tokenizer = tokenizer

    def __call__(self, **kwargs):
        print(f"Received kwargs: {kwargs}")
        print(f"Kwargs keys: {kwargs.keys()}")
        if "messages" in kwargs:
            messages = kwargs.pop("messages")
            prompt = ""
            for msg in messages:
                role = msg.get("role")
                content = msg.get("content", "")
                prompt += f"{role.upper()}:\n{content}\n"
        elif "prompt" in kwargs:
            prompt = kwargs.pop("prompt")
        else:
            raise ValueError("No prompt or messages found in inputs")

        # Tokenize and generate
        max_new_tokens = kwargs.pop("max_new_tokens", 128)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self._hf_model.device)
        outputs = self._hf_model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Model response: {text}")

        return text

In [28]:
hebrew_lm = TinyHebrewLM(model, tokenizer)
dspy.configure(lm=hebrew_lm)

## Remote LM

In [23]:
# lm = dspy.LM('gemini/gemini-2.5-pro-preview-03-25', api_key='')
# dspy.configure(lm=lm)

# Using it

In [29]:
sr_classifier = SRClassifier()
user_prompt = posts_df.iloc[3]['text']
label = sr_classifier(user_prompt)
print(label)

Received kwargs: {'messages': [{'role': 'system', 'content': "Your input fields are:\n1. `text` (str): Hebrew post to classify.\nYour output fields are:\n1. `label` (Literal['suicidal-risk', 'non-suicidal-risk']): Classification result.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## text ## ]]\n{text}\n\n[[ ## label ## ]]\n{label}        # note: the value you produce must exactly match (no extra characters) one of: suicidal-risk; non-suicidal-risk\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Given the fields `text`, produce the fields `label`."}, {'role': 'user', 'content': '[[ ## text ## ]]\nבערך מגיל 17 אני סובלת מחרדה, כאשר חלה החרפה קיצונית בגיל 18 עקב לימודים ומצב חברתי גרוע והשפלות יומיומיות  . בהתחלה לא הבנתי בכלל שמדובר בחרדה כי הסימנים היו פיזיולוגים והחששות תמיד היו. התחלתי "להתעלף"  , הרגשתי שאני לא מצליחה לנשום וגם כשאני כן מצליחה אז האוויר לא נכנס לי לריאות , היו לי סחרחורו

AdapterParseError: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: S 

Expected to find output fields in the LM response: [label] 



Problem:
the model doesnt obey instructions!!

In [ ]:
# sr_classifier = SRClassifier()
# posts_labels = []
# i = 0
# for index, row in posts_df.iterrows():
#     if i > 0: break
#     question_id = row['question_id']
#     text = row['text']

#     label = sr_classifier(text)
#     posts_labels.append({'question_id': question_id, 'label': label})
#     i += 1

# print(f"Classification of {len(posts_labels)} posts completed.")

In [ ]:
# post_labels_df = pd.DataFrame(posts_labels)
# post_labels_df.head()
# save_path = f'./hafifot-tiug/data/labels/present/{MODEL_SHORT_NAME}'
# pd.to_csv(f'{save_path}.csv', index=False)
# print(f"{MODEL_SHORT_NAME} labels saved to {save_path} successfully.")